# Text Search within Images — Colab / H100 Training

**Search a collection of images by the TEXT they CONTAIN (OCR).** Pipeline: OCR each image → per-image text → hybrid index (BM25 exact + dense semantic → RRF) → query → rank images + verify the literal term + highlight the matching snippet, and **abstain** when no image contains the text. The trainable core is a dense text retriever fine-tuned with MNRL.

**One button:** run the *Setup* cells, then **ONE-BUTTON AUTOPILOT**. It auto-detects the GPU (H100 → A100 → L4 → T4), fine-tunes the retriever, evaluates vs baselines, runs analysis, and writes `report.pdf` + `slides.pptx` + a submission bundle to your Drive.

_Author: Le Dinh Minh Quan (23127460) — NLP in Industry, Final Assignment (P20)._

## 0. Controls

In [ ]:
#@title Controls { run: 'auto' }
USE_DRIVE = True           #@param {type:'boolean'}
CLONE_FROM_GIT = True      #@param {type:'boolean'}
GIT_URL = 'https://github.com/ledinhminhquan/20_Text_Search_in_Images.git'  #@param {type:'string'}
TRAIN_RETRIEVER = True     #@param {type:'boolean'}
USE_REAL_DATA = False      #@param {type:'boolean'}   # index a real HF collection instead of synthetic
EPOCHS = 2                 #@param {type:'integer'}
PAIR_LIMIT = 8000          #@param {type:'integer'}
RUN_AUTOPILOT = True       #@param {type:'boolean'}
print('controls set')

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo 'No GPU — runtime > Change runtime type > GPU (H100/A100/L4/T4)'
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
                    '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Drive mount + paths

In [ ]:
import os
if USE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/imgtextsearch'
else:
    BASE = '/content/imgtextsearch'
os.makedirs(BASE, exist_ok=True)
os.environ['IMGTEXT_ARTIFACTS_DIR'] = BASE + '/artifacts'
os.environ['HF_HOME'] = BASE + '/hf'
os.environ['IMGTEXT_USE_HF'] = '1' if USE_REAL_DATA else '0'
print('artifacts ->', os.environ['IMGTEXT_ARTIFACTS_DIR'])

## 3. Get the code

In [ ]:
import os
if CLONE_FROM_GIT:
    if not os.path.isdir('/content/repo'):
        !git clone $GIT_URL /content/repo
    else:
        !cd /content/repo && git pull --ff-only || true
    PROJ = '/content/repo'
else:
    # Upload/extract the repo zip to Drive and set the path below.
    PROJ = BASE + '/20_Text_Search_in_Images'
os.environ['PROJ'] = PROJ
assert os.path.isdir(PROJ + '/src/imgtextsearch'), 'repo not found at ' + PROJ
print('repo at', PROJ)

## 4. Install (Colab-safe)
Install the ML/serving/report deps from `requirements_colab.txt` (torch is preinstalled on Colab), then the package with `--no-deps` so it does not perturb Colab's resolved torch/CUDA. Tesseract is apt-installed for real OCR.

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null && echo 'tesseract installed'
!pip -q install -r $PROJ/requirements_colab.txt
!pip -q install -e $PROJ --no-deps
import importlib, imgtextsearch; importlib.reload(imgtextsearch)
print('imgtextsearch', imgtextsearch.__version__)

## 5. GPU auto-profile → write `train_colab.yaml`
Batch size auto-scales by GPU tier (H100 → A100 → L4 → T4).

In [ ]:
import torch, yaml, os
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
mem = (torch.cuda.get_device_properties(0).total_memory/1e9) if torch.cuda.is_available() else 0
if 'H100' in name: bs, tf32, bf16 = 192, True, True
elif 'A100' in name: bs, tf32, bf16 = 128, True, True
elif 'L4' in name:  bs, tf32, bf16 = 64, True, True
elif 'T4' in name:  bs, tf32, bf16 = 32, False, False
else:               bs, tf32, bf16 = 16, False, False
cfg = {'data': {'use_hf': bool(USE_REAL_DATA), 'collection_size': 400, 'seed': 42},
       'model': {'base_model': 'BAAI/bge-small-en-v1.5', 'num_train_epochs': int(EPOCHS),
                 'per_device_train_batch_size': bs, 'bf16': bf16, 'tf32': tf32, 'max_seq_length': 256},
       'index': {'use_bm25': True, 'use_dense': True, 'use_faiss': True},
       'agent': {'llm_fallback_enabled': False}}
os.makedirs(PROJ + '/configs', exist_ok=True)
open(PROJ + '/configs/train_colab.yaml','w').write(yaml.safe_dump(cfg, sort_keys=False))
print(f'GPU={name} mem={mem:.0f}GB -> batch_size={bs} bf16={bf16} tf32={tf32}')
print(open(PROJ + '/configs/train_colab.yaml').read())

## 6. Sanity-check + render a synthetic collection

In [ ]:
!cd $PROJ && imgtextsearch --config configs/train_colab.yaml data
!cd $PROJ && imgtextsearch --config configs/train_colab.yaml gen-synthetic

## 7. ONE-BUTTON AUTOPILOT 🚀
data → baseline → **train retriever** → evaluate → tune → error-analysis → search-quality → benchmark → demo → monitoring → **report.pdf + slides.pptx** → grade → zipped bundle. Each step is isolated; training is skipped automatically if no GPU is present.

In [ ]:
import os
flag = '' if TRAIN_RETRIEVER else '--no-train'
lim = f'--limit {PAIR_LIMIT}' if PAIR_LIMIT else ''
if RUN_AUTOPILOT:
    !cd $PROJ && imgtextsearch --config configs/train_colab.yaml autopilot $flag $lim
else:
    print('RUN_AUTOPILOT is off — use the individual steps below.')

## 8. Individual steps (optional)

In [ ]:
# Fine-tune only:
# !cd $PROJ && imgtextsearch --config configs/train_colab.yaml train-retriever --limit $PAIR_LIMIT
# Evaluate (full, with the trained dense retriever):
# !cd $PROJ && imgtextsearch --config configs/train_colab.yaml evaluate
# OCR-noise robustness sweep:
# !cd $PROJ && imgtextsearch --config configs/train_colab.yaml tune
# Report + slides only:
# !cd $PROJ && imgtextsearch --config configs/train_colab.yaml generate-report
# !cd $PROJ && imgtextsearch --config configs/train_colab.yaml generate-slides

## 9. Diagnostics

In [ ]:
!cd $PROJ && imgtextsearch --config configs/train_colab.yaml grade
!cd $PROJ && imgtextsearch --config configs/train_colab.yaml demo-agent

## 10. Test the trained retriever

In [ ]:
from imgtextsearch.config import load_config
from imgtextsearch.agent.search_agent import SearchAgent
cfg = load_config(PROJ + '/configs/train_colab.yaml')
agent = SearchAgent(cfg, load_model=True)  # loads the fine-tuned dense retriever if trained
for q in ['invoice', 'REF3386', '2023', 'zzqwx']:
    out = agent.search(q)
    print(q, '->', [(r['id'], round(r['score'],2), r['exact_match']) for r in out['results'][:3]],
          '| abstained=', out['abstained'])

## 11. Locate deliverables

In [ ]:
import glob, os
root = os.environ['IMGTEXT_ARTIFACTS_DIR']
for pat in ['submission/*/report.pdf','submission/*/slides.pptx','submission/*/submission_bundle.zip',
            'runs/*/eval.json','models/*']:
    for p in glob.glob(os.path.join(root, pat)):
        print(p)
print('\nDownload report.pdf + slides.pptx + submission_bundle.zip from the path above (in your Drive).')